In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion4-ml-distribuido-regresion-tanques")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark

In [4]:
import os

# ¿Dónde está corriendo el kernel?
print("CWD actual:", os.getcwd())

# ¿Qué hay en el CWD?
print("\nContenido del CWD:")
for f in os.listdir("."):
    print(" ", f)

# ¿Qué hay en el padre?
print("\nContenido del padre:")
for f in os.listdir(".."):
    print(" ", f)

CWD actual: /home/jovyan

Contenido del CWD:
  .ipynb_checkpoints
  .ipython
  .jupyter
  .local
  .npm
  compose.yml
  data
  Dockerfile
  ml-distribuido-regresion.ipynb
  S02_Equipo_Horeb.ipynb

Contenido del padre:
  jovyan


In [16]:
# Rutas del proyecto
RUTA      = "data/tanques_horeb_muestra.csv"
ARTIFACTS = "artifacts"          # se creará automáticamente al guardar

import os
os.makedirs(ARTIFACTS, exist_ok=True)
print(f"ARTIFACTS = {os.path.abspath(ARTIFACTS)}")

ARTIFACTS = /home/jovyan/artifacts


In [5]:
# El CSV debe estar en la misma carpeta que este notebook
RUTA = "data/tanques_horeb_muestra.csv"

df = spark.read.csv(RUTA, header=True, inferSchema=True)
df.printSchema()

root
 |-- device_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- temp_aire_c: double (nullable = true)
 |-- hum_aire_pct: double (nullable = true)
 |-- nivel_cm_comp: double (nullable = true)
 |-- litros: double (nullable = true)
 |-- capacidad_litros: integer (nullable = true)
 |-- altura_tanque_cm: integer (nullable = true)
 |-- presion_linea_kpa: double (nullable = true)
 |-- caudal_lpm: double (nullable = true)
 |-- temp_agua_c: double (nullable = true)
 |-- orp_mv: double (nullable = true)
 |-- horas_residencia: double (nullable = true)
 |-- bateria_pct: double (nullable = true)



In [8]:
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, DoubleType
)

schema = StructType([
    StructField("device_id",          StringType(),    nullable=False),
    StructField("timestamp",          TimestampType(), nullable=False),
    StructField("zona",               StringType(),    nullable=True),
    StructField("distancia_cm",       DoubleType(),    nullable=True),
    StructField("temp_aire_c",        DoubleType(),    nullable=True),
    StructField("hum_aire_pct",       DoubleType(),    nullable=True),
    StructField("nivel_cm_comp",      DoubleType(),    nullable=True),
    StructField("litros",             DoubleType(),    nullable=True),
    StructField("capacidad_litros",   DoubleType(),    nullable=True),
    StructField("altura_tanque_cm",   DoubleType(),    nullable=True),
    StructField("presion_linea_kpa",  DoubleType(),    nullable=True),
    StructField("caudal_lpm",         DoubleType(),    nullable=True),
    StructField("temp_agua_c",        DoubleType(),    nullable=True),
    StructField("orp_mv",             DoubleType(),    nullable=True),
    StructField("horas_residencia",   DoubleType(),    nullable=True),
    StructField("bateria_pct",        DoubleType(),    nullable=True),
])

df_raw = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
    .schema(schema)
    .csv(RUTA)
)

print(f"Filas: {df_raw.count():,}")
df_raw.printSchema()

Filas: 34,560
root
 |-- device_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- zona: string (nullable = true)
 |-- distancia_cm: double (nullable = true)
 |-- temp_aire_c: double (nullable = true)
 |-- hum_aire_pct: double (nullable = true)
 |-- nivel_cm_comp: double (nullable = true)
 |-- litros: double (nullable = true)
 |-- capacidad_litros: double (nullable = true)
 |-- altura_tanque_cm: double (nullable = true)
 |-- presion_linea_kpa: double (nullable = true)
 |-- caudal_lpm: double (nullable = true)
 |-- temp_agua_c: double (nullable = true)
 |-- orp_mv: double (nullable = true)
 |-- horas_residencia: double (nullable = true)
 |-- bateria_pct: double (nullable = true)



In [9]:
from pyspark.sql.functions import col, count as spark_count, when

# Conteo de nulos por columna
df_raw.select([
    spark_count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
]).show(vertical=True, truncate=False)

-RECORD 0------------------
 device_id         | 0     
 timestamp         | 0     
 zona              | 0     
 distancia_cm      | 0     
 temp_aire_c       | 0     
 hum_aire_pct      | 0     
 nivel_cm_comp     | 207   
 litros            | 207   
 capacidad_litros  | 0     
 altura_tanque_cm  | 0     
 presion_linea_kpa | 30240 
 caudal_lpm        | 30240 
 temp_agua_c       | 30240 
 orp_mv            | 30240 
 horas_residencia  | 30240 
 bateria_pct       | 0     



In [10]:
# Conteo de nulos SEGMENTADO por device_id — el hallazgo clave
df_raw.groupBy("device_id").agg(
    *[spark_count(when(col(c).isNull(), c)).alias(c)
      for c in ["presion_linea_kpa", "caudal_lpm", "temp_agua_c",
                "orp_mv", "horas_residencia", "litros"]]
).show(truncate=False)

+---------------+-----------------+----------+-----------+------+----------------+------+
|device_id      |presion_linea_kpa|caudal_lpm|temp_agua_c|orp_mv|horas_residencia|litros|
+---------------+-----------------+----------+-----------+------+----------------+------+
|horeb_master_01|0                |0         |0          |0     |0               |29    |
|horeb_n08      |4320             |4320      |4320       |4320  |4320            |23    |
|horeb_n04      |4320             |4320      |4320       |4320  |4320            |26    |
|horeb_n05      |4320             |4320      |4320       |4320  |4320            |33    |
|horeb_n06      |4320             |4320      |4320       |4320  |4320            |28    |
|horeb_n07      |4320             |4320      |4320       |4320  |4320            |20    |
|horeb_n02      |4320             |4320      |4320       |4320  |4320            |25    |
|horeb_n03      |4320             |4320      |4320       |4320  |4320            |23    |
+---------

In [11]:
VARIABLES_NUMERICAS = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct", "litros",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
df_raw.select(VARIABLES_NUMERICAS).describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+
|summary|      distancia_cm|       temp_aire_c|      hum_aire_pct|            litros| presion_linea_kpa|         caudal_lpm|       temp_agua_c|            orp_mv|  horas_residencia|        bateria_pct|
+-------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+
|  count|             34560|             34560|             34560|             34353|              4320|               4320|              4320|              4320|              4320|              34560|
|   mean| 32.08764154038251|12.999168981481482| 93.19020543981485|  791.569440805752|17.703981481481488|0.43023148148148166|12.976537037037028| 557.9788425925932|11.974513888888833|  99.922245

In [12]:
PREDICTORES_CANDIDATOS = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
for c in PREDICTORES_CANDIDATOS:
    r = df_raw.stat.corr(c, "litros")
    print(f"{c:22s} correlacion con litros: {r:+.4f}")

distancia_cm           correlacion con litros: -0.7466
temp_aire_c            correlacion con litros: +0.3492
hum_aire_pct           correlacion con litros: -0.3309
presion_linea_kpa      correlacion con litros: +0.0380
caudal_lpm             correlacion con litros: +0.0015
temp_agua_c            correlacion con litros: -0.1789
orp_mv                 correlacion con litros: -0.1384
horas_residencia       correlacion con litros: -0.3042
bateria_pct            correlacion con litros: +0.0750


In [13]:
from pyspark.sql.functions import col

df_master = df_raw.filter(col("device_id") == "horeb_master_01")

df_limpio = df_master.filter(
    (col("litros") >= 0) &
    (col("distancia_cm") > 0) &
    (col("caudal_lpm") >= 0) &
    (col("presion_linea_kpa") >= 0)
)

print(f"Antes: {df_master.count():,}  Despues: {df_limpio.count():,}")

Antes: 4,320  Despues: 4,291


In [14]:
# Duplicados por timestamp
total = df_limpio.count()
sin_dup = df_limpio.dropDuplicates(["timestamp"]).count()
print(f"Total: {total:,}, sin duplicar por timestamp: {sin_dup:,}")
assert total == sin_dup, "Hay timestamp duplicado"

# Nulos en variables del modelo
VARIABLES_MODELO = [
    "litros",
    "distancia_cm", "temp_aire_c", "hum_aire_pct",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
antes  = df_limpio.count()
df_valido = df_limpio.na.drop(subset=VARIABLES_MODELO)
despues = df_valido.count()
print(f"Filas antes: {antes:,}  despues: {despues:,}  eliminadas: {antes-despues:,}")

df_valido = df_valido.cache()

Total: 4,291, sin duplicar por timestamp: 4,291
Filas antes: 4,291  despues: 4,291  eliminadas: 0


In [17]:
from pyspark.sql.functions import date_format

df_particionable = df_valido.withColumn(
    "AnioMes", date_format(col("timestamp"), "yyyy-MM")
)

(
    df_particionable
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("zona", "AnioMes")
    .save(f"{ARTIFACTS}/tanques_particionado")
)

import os
for carpeta in sorted(os.listdir(f"{ARTIFACTS}/tanques_particionado")):
    print(carpeta)

._SUCCESS.crc
_SUCCESS
zona=Zona A


In [18]:
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/tanques_particionado")
assert df_verificacion.count() == df_particionable.count()
print(f"Verificado: {df_verificacion.count():,} filas")

df_verificacion.filter(col("zona") == "Zona A").explain(True)
# Debes ver PartitionFilters: [..., (zona = Zona A)]

Verificado: 4,291 filas
== Parsed Logical Plan ==
'Filter ('zona = Zona A)
+- Relation [device_id#3732,timestamp#3733,distancia_cm#3734,temp_aire_c#3735,hum_aire_pct#3736,nivel_cm_comp#3737,litros#3738,capacidad_litros#3739,altura_tanque_cm#3740,presion_linea_kpa#3741,caudal_lpm#3742,temp_agua_c#3743,orp_mv#3744,horas_residencia#3745,bateria_pct#3746,zona#3747,AnioMes#3748] parquet

== Analyzed Logical Plan ==
device_id: string, timestamp: timestamp, distancia_cm: double, temp_aire_c: double, hum_aire_pct: double, nivel_cm_comp: double, litros: double, capacidad_litros: double, altura_tanque_cm: double, presion_linea_kpa: double, caudal_lpm: double, temp_agua_c: double, orp_mv: double, horas_residencia: double, bateria_pct: double, zona: string, AnioMes: string
Filter (zona#3747 = Zona A)
+- Relation [device_id#3732,timestamp#3733,distancia_cm#3734,temp_aire_c#3735,hum_aire_pct#3736,nivel_cm_comp#3737,litros#3738,capacidad_litros#3739,altura_tanque_cm#3740,presion_linea_kpa#3741,caudal

In [19]:
PREDICTORES = [
    "distancia_cm", "temp_aire_c", "hum_aire_pct",
    "presion_linea_kpa", "caudal_lpm", "temp_agua_c",
    "orp_mv", "horas_residencia", "bateria_pct",
]
print(f"Predictores ({len(PREDICTORES)}): {PREDICTORES}")

Predictores (9): ['distancia_cm', 'temp_aire_c', 'hum_aire_pct', 'presion_linea_kpa', 'caudal_lpm', 'temp_agua_c', 'orp_mv', 'horas_residencia', 'bateria_pct']


In [20]:
from pyspark.ml.feature import VectorAssembler

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_ml = ensamblador.transform(df_verificacion).select("features", "litros")
df_ml.show(5, truncate=False)

+-------------------------------------------------+------+
|features                                         |litros|
+-------------------------------------------------+------+
|[7.31,14.57,90.5,0.0,0.0,9.05,633.3,0.67,99.8]   |928.0 |
|[22.13,26.29,90.0,0.0,0.0,15.54,574.8,8.67,100.0]|776.0 |
|[13.84,27.47,89.9,0.0,0.0,12.87,591.7,5.17,100.0]|860.0 |
|[32.05,16.4,92.5,0.0,0.0,16.91,551.0,11.67,100.0]|682.0 |
|[42.45,8.48,94.1,0.0,0.0,15.83,545.0,13.83,99.9] |584.0 |
+-------------------------------------------------+------+
only showing top 5 rows



In [21]:
df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Entrenamiento: {df_train.count():,}  Prueba: {df_test.count():,}")

Entrenamiento: 3,430  Prueba: 861


In [22]:
from pyspark.ml.regression import LinearRegression

lr_base = LinearRegression(featuresCol="features", labelCol="litros")
modelo_base = lr_base.fit(df_train)
print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)

Coeficientes: [-4.288791492014177,0.2426944111998363,-0.3827104140800306,-0.07432066497601166,-6.798477720329463,-2.1002050323986268,0.9188153391703686,-7.463354801368742,-131.62750873888268]
Intercepto: 13611.164448850814


In [23]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluar(predicciones, nombre):
    res = {}
    for m in ["rmse", "r2", "mae"]:
        ev = RegressionEvaluator(labelCol="litros",
                                 predictionCol="prediction",
                                 metricName=m)
        res[m.upper()] = ev.evaluate(predicciones)
    print(f"{nombre}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}  MAE={res['MAE']:.4f}")
    return res

pred_base = modelo_base.transform(df_test)
res_base  = evaluar(pred_base, "LinearRegression base")

LinearRegression base: RMSE=37.8487  R2=0.9735  MAE=23.3145


In [25]:
import pandas as pd

configuraciones = [
    {"nombre": "Sin regularizacion",   "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)",           "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)",  "regParam": 0.1, "elasticNetParam": 0.5},
]

comparacion_configs = []
for cfg in configuraciones:
    lr = LinearRegression(featuresCol="features", labelCol="litros",
                          regParam=cfg["regParam"],
                          elasticNetParam=cfg["elasticNetParam"])
    m = lr.fit(df_train)
    r = evaluar(m.transform(df_test), cfg["nombre"])
    r["Configuracion"] = cfg["nombre"]
    comparacion_configs.append(r)

pd.DataFrame(comparacion_configs)[["Configuracion", "RMSE", "R2", "MAE"]]

Sin regularizacion: RMSE=37.8487  R2=0.9735  MAE=23.3145
Ridge (L2): RMSE=37.8830  R2=0.9735  MAE=23.3504
Elastic Net (L1+L2): RMSE=37.8555  R2=0.9735  MAE=23.3083


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,37.848696,0.973527,23.314460
1,Ridge (L2),37.883024,0.973479,23.350400
2,Elastic Net (L1+L2),37.855460,0.973517,23.308279


In [26]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol="litros",
                           numTrees=50, maxDepth=8, seed=42)
modelo_rf = rf.fit(df_train)
res_rf = evaluar(modelo_rf.transform(df_test), "Random Forest")

Random Forest: RMSE=13.2713  R2=0.9967  MAE=7.3084


In [27]:
importancias = sorted(
    zip(PREDICTORES, modelo_rf.featureImportances.toArray()),
    key=lambda x: x[1], reverse=True
)
for v, imp in importancias:
    print(f"{v:22s} {imp:.4f}")

horas_residencia       0.3856
distancia_cm           0.2689
orp_mv                 0.2335
temp_aire_c            0.0534
bateria_pct            0.0264
hum_aire_pct           0.0144
temp_agua_c            0.0135
presion_linea_kpa      0.0027
caudal_lpm             0.0017


In [28]:
comparacion_final = pd.DataFrame(
    comparacion_configs + [{**res_rf, "Configuracion": "Random Forest"}]
)[["Configuracion", "RMSE", "R2", "MAE"]]

comparacion_final.sort_values("RMSE")

,Configuracion,RMSE,R2,MAE
3,Random Forest,13.271293,0.996745,7.308439
0,Sin regularizacion,37.848696,0.973527,23.314460
2,Elastic Net (L1+L2),37.855460,0.973517,23.308279
1,Ridge (L2),37.883024,0.973479,23.350400


In [29]:
modelo_ganador = modelo_rf   # ajusta según tu resultado
modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_litros_rf")
print(f"Modelo guardado en {ARTIFACTS}/modelo_litros_rf")

Modelo guardado en artifacts/modelo_litros_rf
